# 01. Baseline Characteristics (Table 1)

Reproduce Table 1 of the manuscript — demographic and urine dipstick distributions for the development cohort and the KNHANES external validation cohort, stratified by FPG- and HbA1c-based analytic samples.

**Inputs required**:
- `data/processed/development_cohort.csv` (hospital EMR — not publicly distributed; available via collaborative request)
- `data/processed/knhanes_external.csv` (KNHANES 2016–2024 — preparation via `data/knhanes_processing.py`)

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))

import pandas as pd
import numpy as np
from data_utils import BASE_FEATURES, load_data

DEV_PATH = '../data/processed/development_cohort.csv'   # not committed
KN_PATH  = '../data/processed/knhanes_external.csv'      # produced by data/knhanes_processing.py
TIME_WINDOW = 14

## Development cohort — load & filter

In [ ]:
dev = load_data(DEV_PATH)
dev = dev[dev['days_difference'].abs() <= TIME_WINDOW]
fpg_dev   = dev[BASE_FEATURES + ['serum_uric_acid']].dropna(subset=BASE_FEATURES + ['serum_uric_acid'])
serum_uric_acid_dev = dev[BASE_FEATURES + ['serum_uric_acid']].dropna(subset=BASE_FEATURES + ['serum_uric_acid'])
print(f'FPG sample:   n = {len(fpg_dev):,}')
print(f'HbA1c sample: n = {len(serum_uric_acid_dev):,}')

## KNHANES external cohort

In [ ]:
kn = pd.read_csv(KN_PATH, low_memory=False)
fpg_kn   = kn[BASE_FEATURES + ['HE_glu']].dropna(subset=BASE_FEATURES + ['HE_glu'])
serum_uric_acid_kn = kn[BASE_FEATURES + ['HE_HbA1c']].dropna(subset=BASE_FEATURES + ['HE_HbA1c'])
print(f'KNHANES FPG sample:   n = {len(fpg_kn):,}')
print(f'KNHANES HbA1c sample: n = {len(serum_uric_acid_kn):,}')

## Build Table 1 (demographics + dipstick + outcomes)

In [ ]:
def summarize(df, lab_col, lab_thr):
    s = {
        'n': len(df),
        'age_mean': df['age'].mean(), 'age_sd': df['age'].std(),
        'male_n': int(df['male'].sum()), 'male_pct': df['male'].mean()*100,
        'pH_mean': df['ua_pH'].mean(), 'pH_sd': df['ua_pH'].std(),
        'sg_mean': df['ua_sg'].mean(), 'sg_sd': df['ua_sg'].std(),
    }
    for c in ['ua_glucose','ua_protein','ua_ketone','ua_bilirubin','ua_blood','ua_urobilinogen']:
        pos = (df[c] > 0).sum()
        s[f'{c}_pos_n'] = int(pos)
        s[f'{c}_pos_pct'] = pos/len(df)*100
    s['nitrite_pos'] = int((df['ua_nitrite'] >= 1).sum())
    pos = (df[lab_col] >= lab_thr).sum()
    s[f'outcome_n'] = int(pos); s['outcome_pct'] = pos/len(df)*100
    return s

rows = {
    'Dev FPG (n=491,856)':   summarize(fpg_dev,   'serum_uric_acid',  126),
    'Dev HbA1c (n=152,304)': summarize(serum_uric_acid_dev, 'serum_uric_acid',    6.5),
    'KNHANES FPG':           summarize(fpg_kn,    'HE_glu',   126),
    'KNHANES HbA1c':         summarize(serum_uric_acid_kn,  'HE_HbA1c', 6.5),
}
pd.DataFrame(rows).T